# Parameter-Efficient Fine-Tuning (PEFT)

A practical reference for adapting large pretrained models to new tasks by training a **tiny fraction** of the weights instead of all of them. Covers the family of methods (LoRA, QLoRA, adapters, prefix/prompt tuning, IA³, DoRA), why they work, how to run them with Hugging Face `peft`, memory math, serving multiple adapters, and the trade-offs against full fine-tuning.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**Parameter-Efficient Fine-Tuning (PEFT)** is a family of techniques that adapt a large pretrained model to a downstream task by training only a **small set of new or selected parameters** while keeping the original weights frozen. A 7B-parameter model has ~7 billion trainable weights; with LoRA you might train only **4–20 million** of them — under 0.3% — and reach quality within a point or two of full fine-tuning on most tasks.

### What is it?

PEFT inserts small trainable modules (or selects a small slice of existing parameters) into a frozen backbone. The backbone provides the general capability learned during pretraining; the small PEFT module learns the task- or domain-specific delta. Because the frozen weights never change, you can:

- Store a task as a few megabytes of adapter weights instead of a full model copy.
- Train on a single consumer/prosumer GPU that could never hold full-fine-tuning optimizer state.
- Hot-swap many adapters on top of one shared base model at serving time.

### Why use it?

Key benefits of PEFT:

- **Drastically lower memory.** Full fine-tuning of a 7B model in mixed precision needs roughly weights + gradients + Adam moments ≈ `7B × (2 + 2 + 8) ≈ 84 GB`. LoRA trains only the adapter, so gradients and optimizer state shrink by ~99%, fitting on a 24 GB (or, with QLoRA, a 12–16 GB) GPU.
- **Tiny, composable artifacts.** An adapter is typically 2–200 MB. You can keep hundreds of them and load the right one per request.
- **No catastrophic forgetting of the base.** The frozen backbone is shared and unchanged, so general capabilities are preserved and adapters stay isolated from each other.
- **Faster iteration and cheaper storage.** Less to checkpoint, version, and ship.

### When to use it?

PEFT is particularly useful when:

- You need to specialize a large model to a domain or task but cannot afford full-fine-tuning compute/memory.
- You must support **many** tasks/tenants/customers from **one** base model (multi-tenant serving).
- Your task data is small-to-medium (hundreds to low-millions of examples) — full fine-tuning would overfit or is overkill.
- You want quick experimentation cycles and small, easily-versioned artifacts.

Reach for **full fine-tuning** instead when you need to deeply change the model's behavior/knowledge, have abundant data and compute, or are doing continued pretraining.

## Key Features

### Core PEFT methods and what each one does

| Method | What it trains | Benefit |
|--------|----------------|---------|
| **LoRA** (Low-Rank Adaptation) | Two small matrices `A` (d×r) and `B` (r×d) whose product is added to a frozen weight `W`; only `A`,`B` train | Best general-purpose default; ~0.1–1% trainable params, mergeable into `W` for zero inference overhead |
| **QLoRA** | LoRA on top of a 4-bit (NF4) quantized frozen base | Fine-tune a 65B model on a single 48 GB GPU; biggest memory win |
| **Adapters** (Houlsby/Pfeiffer) | Small bottleneck MLP modules inserted between transformer sublayers | Strong modularity; adds a little inference latency unless fused |
| **Prefix / P-Tuning v2** | Trainable key/value vectors prepended at every attention layer | No change to model weights; good for generation tasks |
| **Prompt Tuning** | A handful of trainable "soft prompt" embeddings prepended to the input | Smallest footprint (kilobytes); works best at large model scale |
| **IA³** | Learned scaling vectors that rescale keys, values, and FFN activations | Even fewer params than LoRA; element-wise, cheap |
| **DoRA** (Weight-Decomposed LoRA) | Decomposes weights into magnitude + direction, applies LoRA to direction | Closes more of the gap to full fine-tuning at similar param count |

LoRA is the right default for most people; QLoRA when memory is tight; prompt/prefix tuning when you must not touch weights at all.

## Architecture Overview

LoRA is the canonical example. For a frozen linear layer `y = W x` (with `W` of shape `d_out × d_in`), LoRA learns a **low-rank update** `ΔW = (α/r) · B A`, where `A` is `r × d_in`, `B` is `d_out × r`, and the rank `r` is small (4–64). The forward pass becomes:

```
                 ┌──────────────────────────────┐
                 │      frozen pretrained W      │   (no gradient)
       x ───────►│           y = W·x             ├────┐
                 └──────────────────────────────┘    │
                                                      ▼
       x ──► [ A: r×d_in ] ──► [ B: d_out×r ] ──► (α/r)·B·A·x ──►(+)──► y'
              (trainable)        (trainable)                       output

  y' = W·x + (α/r)·B·A·x          A init ~N(0,σ),  B init = 0  → ΔW starts at 0
```

Because `B` is initialized to zero, training begins exactly at the pretrained behavior and only departs as needed. `r` controls capacity; `α` (alpha) is a scaling factor — the effective scale is `α/r`.

### Components

1. **Frozen backbone.** The original pretrained weights, kept in eval/`requires_grad=False`. In QLoRA these are stored in 4-bit NF4.
2. **PEFT modules.** The small trainable tensors (LoRA `A`/`B`, adapter MLPs, soft-prompt embeddings, IA³ scalers). Only these receive gradients and live in the optimizer.
3. **Injection points.** Which sublayers get adapted — for LoRA, typically the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and often the MLP (`gate_proj`, `up_proj`, `down_proj`).
4. **Adapter store + merge step.** Adapters are saved separately (`adapter_model.safetensors` + `adapter_config.json`). At deploy time you either keep them separate (hot-swappable) or **merge** `ΔW` back into `W` for a standalone model with zero added latency.

## Installation

### Prerequisites

- Python 3.9+
- PyTorch with CUDA for real training (CPU works for the toy NumPy demo below)
- An NVIDIA GPU for anything beyond tiny models; 16 GB+ recommended for 7B QLoRA
- For 4-bit QLoRA: a CUDA GPU (bitsandbytes requires it)

### Installation Steps

**Note**: Uncomment the following cell to install the Hugging Face PEFT stack.

In [ ]:
# Uncomment to install the Hugging Face PEFT stack.
# %pip install -U peft transformers accelerate datasets
# QLoRA (4-bit) additionally needs bitsandbytes on a CUDA machine:
# %pip install -U bitsandbytes

## Basic Usage

### Quick Start Example

The mechanics of LoRA are simple enough to implement in a few lines of NumPy — no GPU, no downloads. This makes the math concrete: a frozen weight plus a low-rank trainable delta.

In [ ]:
# A from-scratch LoRA forward pass in pure NumPy — runs anywhere, no deps beyond numpy.
import numpy as np

rng = np.random.default_rng(0)

d_in, d_out, r, alpha = 16, 8, 4, 8
W = rng.normal(size=(d_out, d_in))          # frozen pretrained weight (never updated)

# LoRA factors: A ~ small random, B = 0  ->  initial delta is exactly zero.
A = rng.normal(scale=0.01, size=(r, d_in))  # trainable
B = np.zeros((d_out, r))                     # trainable

def forward(x, A, B):
    base = W @ x                             # frozen path
    delta = (alpha / r) * (B @ (A @ x))      # low-rank adapter path
    return base + delta

x = rng.normal(size=(d_in,))
print("delta at init is zero:", np.allclose(forward(x, A, B), W @ x))

# Param accounting: full fine-tune would update W (d_out*d_in); LoRA updates A+B.
full = d_out * d_in
lora = A.size + B.size
print(f"full-FT params: {full}   LoRA params: {lora}   ratio: {lora/full:.1%}")

In [ ]:
# The same idea with Hugging Face `peft` on a real model.
# Gated behind a try/except so the notebook runs even without the libraries installed.
try:
    import torch
    from transformers import AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model, TaskType

    base = AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")  # ~1M params, CPU-friendly

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["c_attn"],   # GPT-2 fuses q/k/v into c_attn
        bias="none",
    )
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()
    # -> trainable params: ~tiny  ||  all params: ~1.1M  ||  trainable%: <1%
except Exception as e:  # noqa: BLE001 - libraries optional in this environment
    print("peft/transformers not installed — showing API shape only:", type(e).__name__)

## Advanced Features

### Beyond plain LoRA

#### QLoRA: 4-bit base + LoRA on top

QLoRA quantizes the **frozen** base to 4-bit NF4 (NormalFloat) and trains LoRA adapters in higher precision on top. Gradients only flow through the LoRA params, so the 4-bit base never needs full-precision gradients. Two extra tricks: **double quantization** (quantize the quantization constants) and **paged optimizers** (offload optimizer state to CPU on memory spikes). This is what lets a 65B model fine-tune on a single 48 GB GPU.

#### Choosing rank, alpha, and target modules

- **`r` (rank):** 8–16 is a strong default; raise to 32–64 for harder tasks or larger data. Higher `r` = more capacity and more params.
- **`lora_alpha`:** the update is scaled by `alpha/r`. A common convention is `alpha = 2 * r`. With `rslora`, the scale uses `alpha/sqrt(r)` for more stable high-rank behavior.
- **`target_modules`:** adapting **all** linear layers (attention + MLP) usually beats attention-only, at the cost of more params. Use `target_modules="all-linear"` in `peft` to hit them all.

#### DoRA and IA³

- **DoRA** decomposes each weight into a magnitude scalar and a direction matrix, applying LoRA only to the direction. Enable with `use_dora=True` in `LoraConfig`. It typically matches full fine-tuning more closely at the same rank.
- **IA³** learns three rescaling vectors (keys, values, FFN) — far fewer params than LoRA and no rank to tune.

In [ ]:
# QLoRA config sketch: 4-bit NF4 base + LoRA. Needs a CUDA GPU + bitsandbytes to actually load.
try:
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    # On a GPU this loads the frozen base in 4-bit:
    # base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B", quantization_config=bnb_cfg, device_map="auto")
    # base = prepare_model_for_kbit_training(base)   # casts norms to fp32, enables grad checkpointing

    qlora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",   # adapt every linear layer
        use_dora=False,                # flip to True for DoRA
    )
    print("QLoRA config ready:", qlora_cfg.r, qlora_cfg.lora_alpha, bnb_cfg.bnb_4bit_quant_type)
except Exception as e:  # noqa: BLE001
    print("bitsandbytes/transformers not available — config shape shown above:", type(e).__name__)

## Use Cases

### Real-world applications of PEFT

#### Use Case 1: Multi-tenant SaaS — one base, many customer adapters

- **Context:** A platform serves 500 customers, each wanting the model tuned on their own tone/data, but cannot host 500 full model copies.
- **Implementation:** Train one LoRA adapter per customer (a few MB each). Keep one shared base in GPU memory; load the requested adapter per request, or batch requests by adapter. Frameworks like vLLM and LoRAX support serving **hundreds** of LoRA adapters over a single base.
- **Results:** Storage and GPU memory dominated by a single base; per-customer cost is a small adapter file instead of a 14 GB checkpoint.

#### Use Case 2: Domain adaptation on a single GPU (QLoRA)

- **Context:** A team wants to specialize an 8–13B model on legal/medical text but only has a 24 GB GPU.
- **Implementation:** QLoRA — load the base in 4-bit NF4, train rank-16 LoRA on all linear layers with a paged AdamW optimizer and gradient checkpointing.
- **Results:** Fine-tuning that would need 80 GB+ in full precision fits comfortably in 16–24 GB, finishing overnight on commodity hardware.

## Best Practices

### Recommended practices for PEFT

1. **Start with LoRA `r=8–16`, `alpha=2r`, on all linear layers.** This is the modern strong baseline; tune from there rather than from attention-only.
2. **Match precision carefully.** Keep LayerNorm and the LM head in fp32/bf16; use `prepare_model_for_kbit_training` for QLoRA so numerics stay stable.
3. **Use a learning rate ~10× higher than full fine-tuning** (e.g. `1e-4`–`3e-4`). Adapters have far fewer params and tolerate (need) more aggressive LRs.
4. **Enable gradient checkpointing** for long sequences/large models to trade compute for memory; it stacks well with QLoRA.
5. **Save only the adapter** (`model.save_pretrained(dir)` on the PEFT model) — a few MB — and version it alongside the base model **id + revision** so it's reproducible.
6. **Merge for latency-critical serving** (`merge_and_unload()`); keep adapters separate when you need hot-swapping or multi-adapter serving.

In [ ]:
# Saving, reloading, and merging adapters — the typical lifecycle.
try:
    from transformers import AutoModelForCausalLM
    from peft import PeftModel, get_peft_model, LoraConfig, TaskType

    base_id = "sshleifer/tiny-gpt2"
    base = AutoModelForCausalLM.from_pretrained(base_id)
    cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
                     target_modules=["c_attn"], lora_dropout=0.0)
    model = get_peft_model(base, cfg)

    # 1. Save ONLY the adapter (small).
    model.save_pretrained("/tmp/peft-demo-adapter")

    # 2. Reload: fresh base + adapter on top.
    reloaded = AutoModelForCausalLM.from_pretrained(base_id)
    reloaded = PeftModel.from_pretrained(reloaded, "/tmp/peft-demo-adapter")

    # 3. Merge delta into W for a standalone, zero-overhead model.
    merged = reloaded.merge_and_unload()
    print("merged model type:", type(merged).__name__)
except Exception as e:  # noqa: BLE001
    print("peft not installed — lifecycle is: save_pretrained -> PeftModel.from_pretrained -> merge_and_unload:",
          type(e).__name__)

## Common Pitfalls

### What to avoid when using PEFT

1. **Adapting too few modules.** Attention-only LoRA often underperforms; include the MLP / use `all-linear` unless params are truly constrained. *Avoid by* benchmarking against the all-linear baseline.
2. **Wrong `target_modules` names.** They are architecture-specific (`c_attn` for GPT-2, `q_proj`/`v_proj` for Llama). A typo silently adapts nothing and "training" does nothing. *Avoid by* printing `print_trainable_parameters()` and confirming the % is non-zero.
3. **Learning rate too low.** Reusing a full-fine-tune LR (e.g. `2e-5`) makes adapters learn glacially. *Avoid by* starting around `1e-4`–`2e-4`.
4. **Forgetting the base model identity at load time.** An adapter is meaningless without its exact base (same id + revision + quantization). *Avoid by* pinning the base revision and storing it in the adapter config/README.
5. **Merging a QLoRA adapter into a 4-bit base.** Merging needs the base in a mergeable dtype; merge into the **dequantized/fp16** base, not the 4-bit one. *Avoid by* reloading the base in fp16 before `merge_and_unload()`.

## Performance Optimization

### Optimizing PEFT for production

#### The memory math

For a model with `P` parameters fine-tuned with Adam in mixed precision, full fine-tuning needs roughly:

```
weights (fp16)      2P bytes
gradients (fp16)    2P bytes
Adam m, v (fp32)    8P bytes
------------------------------
total              ~12P bytes   (≈84 GB for a 7B model, before activations)
```

LoRA freezes the weights, so gradients and optimizer state apply only to the **adapter** params `P_lora ≪ P`:

```
frozen weights      2P                       (QLoRA: ~0.5P in 4-bit)
adapter grads       2·P_lora
adapter Adam        8·P_lora
```

With `P_lora ≈ 0.3% of P`, the gradient+optimizer term collapses by ~99%. QLoRA shrinks the dominant frozen-weights term too.

#### Key parameters to optimize

- **Rank `r`:** higher rank = more capacity and memory; sweep `{8, 16, 32}`.
- **Quantization (4-bit NF4 vs 8-bit):** 4-bit halves base memory vs 8-bit at a small quality cost.
- **Gradient checkpointing + sequence length:** the activation memory often dominates at long context; checkpointing trades ~30% compute for large memory savings.

In [ ]:
# Estimate training memory: full fine-tuning vs LoRA vs QLoRA, for a given model size.
def mem_gb(num_params_b, method="full", r_frac=0.003):
    P = num_params_b * 1e9
    if method == "full":
        # weights(2) + grads(2) + adam(8) bytes/param
        return P * 12 / 1e9
    if method == "lora":
        adapter = P * r_frac
        return (P * 2 + adapter * (2 + 8)) / 1e9      # frozen fp16 base + adapter grad/optim
    if method == "qlora":
        adapter = P * r_frac
        return (P * 0.5 + adapter * (2 + 8)) / 1e9    # 4-bit base ~0.5 bytes/param
    raise ValueError(method)

for size in (7, 13, 70):
    full = mem_gb(size, "full")
    lora = mem_gb(size, "lora")
    qlora = mem_gb(size, "qlora")
    print(f"{size:>3}B  full={full:6.1f} GB   lora={lora:6.1f} GB   qlora={qlora:6.1f} GB")

## Production Deployment

### Deploying PEFT in production

Two deployment modes:

1. **Merged** — fold `ΔW` into `W` and ship a standalone model. Zero added latency, but you lose hot-swapping and storage per task = full model.
2. **Adapter-on-base** — keep the base loaded once and attach adapters at request time. Slightly more latency unless the server fuses, but enables **multi-adapter** serving (vLLM `--enable-lora`, LoRAX, Hugging Face TGI). Ideal for multi-tenant.

#### Docker Deployment

```dockerfile
FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04
RUN apt-get update && apt-get install -y python3-pip && rm -rf /var/lib/apt/lists/*
RUN pip3 install --no-cache-dir vllm peft transformers
# Bake the merged model, or mount adapters at runtime via a volume.
COPY ./merged-model /models/my-model
EXPOSE 8000
# Serve with adapter hot-loading enabled:
CMD ["python3", "-m", "vllm.entrypoints.openai.api_server", \
     "--model", "/models/base", "--enable-lora", \
     "--lora-modules", "cust-a=/adapters/cust-a", "cust-b=/adapters/cust-b"]
```

#### Kubernetes Deployment

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: peft-llm
spec:
  replicas: 2
  selector:
    matchLabels: {app: peft-llm}
  template:
    metadata:
      labels: {app: peft-llm}
    spec:
      containers:
        - name: server
          image: registry.example.com/peft-llm:1.0.0
          args: ["--model", "/models/base", "--enable-lora"]
          resources:
            limits:
              nvidia.com/gpu: 1
          volumeMounts:
            - {name: adapters, mountPath: /adapters}
          readinessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 60
      volumes:
        - name: adapters
          persistentVolumeClaim: {claimName: lora-adapters}
---
apiVersion: v1
kind: Service
metadata: {name: peft-llm}
spec:
  selector: {app: peft-llm}
  ports: [{port: 80, targetPort: 8000}]
```

## Monitoring and Observability

### Monitoring PEFT in production

#### Key metrics to track

- **Adapter load latency / cache hit rate:** when hot-swapping many adapters, time-to-first-token includes adapter load; track p50/p99 and the in-GPU adapter cache hit ratio.
- **Per-adapter quality:** eval accuracy / win-rate / task metric **per adapter** — a regression in one tenant's adapter shouldn't be hidden by aggregate metrics.
- **Base vs adapter drift:** when you upgrade the base model, every adapter must be re-validated (and often re-trained) against it.
- **GPU memory & throughput:** tokens/sec and KV-cache utilization, since multi-adapter serving competes for the same GPU.

#### Logging best practices

- Log the **base model id + revision and adapter id + version** with every request — reproducibility hinges on this pair.
- Structure logs (JSON) with adapter name, rank, and merge state so you can slice latency/quality by adapter.
- Use appropriate log levels: per-request at DEBUG, adapter load/evict and config at INFO, OOM/quantization fallbacks at WARN/ERROR.

## Troubleshooting

### Common issues with PEFT

#### Issue 1: `print_trainable_parameters()` shows 0% (or training does nothing)

**Symptoms:** Loss never moves; trainable params are ~0.

**Cause:** `target_modules` names don't match this architecture's linear layers, so no LoRA was injected.

**Solution:** Inspect module names (`[n for n, _ in model.named_modules()]`) and set the right targets (`c_attn` for GPT-2, `q_proj`/`k_proj`/`v_proj`/`o_proj` for Llama), or use `target_modules="all-linear"`.

#### Issue 2: CUDA OOM during QLoRA training

**Symptoms:** `torch.cuda.OutOfMemoryError` partway through the first steps.

**Cause:** Activation memory at long sequence length, or optimizer spikes, exceed the GPU.

**Solution:** Enable gradient checkpointing, lower `max_seq_len` or batch size (use gradient accumulation to keep effective batch), and use a **paged** optimizer (`optim="paged_adamw_8bit"`) so spikes page to CPU instead of OOMing.

#### Issue 3: Merged model quality differs from the adapter-on-base model

**Symptoms:** After `merge_and_unload()`, outputs change or degrade.

**Cause:** Merging a QLoRA adapter into a 4-bit base reintroduces quantization error into `ΔW`.

**Solution:** Reload the base in fp16/bf16 (not 4-bit), attach the adapter, then merge — or serve adapter-on-base instead of merging.

## Comparison with Alternatives

### How PEFT compares to other adaptation strategies

| Dimension | LoRA / PEFT | Full Fine-Tuning | Prompt Engineering / RAG |
|-----------|-------------|------------------|--------------------------|
| Trainable params | ~0.1–1% of model | 100% | 0% (no training) |
| GPU memory to train | Low (QLoRA: very low) | Very high | None |
| Artifact size per task | MB | Full model (GB) | None / prompt + index |
| Quality ceiling | Near-full on most tasks | Highest | Limited by base capability |
| Multi-task serving | Excellent (swap adapters) | Poor (one model each) | N/A |
| Changes model knowledge deeply | Limited | Yes | No |

### When to choose PEFT

Choose PEFT when:

- You need real task/domain adaptation but can't afford full-fine-tuning memory or storage.
- You must serve **many** tasks/tenants from one base model.
- Your data is small-to-medium and full fine-tuning would overfit or be wasteful.

Prefer **full fine-tuning** for deep behavior/knowledge changes with ample compute and data; prefer **RAG/prompting** when the base already has the capability and you just need to supply fresh facts or steer format — no training required. See the companion notebooks on Full Parameter Fine-Tuning and Instruction Fine-Tuning.

## Resources

### Official Documentation

- Hugging Face PEFT docs: https://huggingface.co/docs/peft
- PEFT GitHub repository: https://github.com/huggingface/peft
- bitsandbytes (4-bit/8-bit quantization): https://github.com/bitsandbytes-foundation/bitsandbytes

### Papers

- LoRA: Low-Rank Adaptation of Large Language Models — https://arxiv.org/abs/2106.09685
- QLoRA: Efficient Finetuning of Quantized LLMs — https://arxiv.org/abs/2305.14314
- DoRA: Weight-Decomposed Low-Rank Adaptation — https://arxiv.org/abs/2402.09353
- IA³ / (Few-Shot PEFT) — https://arxiv.org/abs/2205.05638

### Tutorials and Guides

- Hugging Face: LoRA conceptual guide — https://huggingface.co/docs/peft/conceptual_guides/lora
- QLoRA fine-tuning walkthrough — https://huggingface.co/blog/4bit-transformers-bitsandbytes
- vLLM multi-LoRA serving — https://docs.vllm.ai/en/latest/features/lora.html

### Community Resources

- Hugging Face forums — https://discuss.huggingface.co/
- LoRAX (multi-adapter serving) — https://github.com/predibase/lorax
- Stack Overflow tag — https://stackoverflow.com/questions/tagged/huggingface-transformers

### Related Technologies

- Full Parameter Fine-Tuning (companion notebook)
- Instruction Fine-Tuning / RLHF
- Model quantization (GPTQ, AWQ, NF4) and distillation